# Data sense making (2)

We continue working on EDA stages applied to different types of analyses, geographical and topical analysis.

# Geo-spatial analysis

Photo archives document artworks held by museums and institutions all over the world. But **where are those museums located**? Do different photo archives
tend to document different geographic areas?

Geo-spatial analysis may involve simply placing data points on a geographical map as you would do on another coordinates system (e.g. cartesian axes). In this way, some superficial patterns may surface, which could be further explored to understand whether they are meaningful or not.

💡 We are going to **plot the repositories** described in artresearch (museums, collection holders, galleries, etc) and show the distribution of artworks in those. We expect a map to show us the geographical network of archival collections and institutions they document, therefore showing the general interest of archives patrons in artworks collected in those areas (regardless of the artwork actual provenance).

More precisely, geo-spatial analysis can reveal interesting patterns that turn out having a strong (co)relation with geographical boundaries.

💡 We will **compare the geographic footprint of each archive**, plotting where the artworks they document are, therefore showing whether archives document geographically different areas, complementing each other or overlapping.

## 💡 Plot distribution of artworks on a map grouped by repository

In artresearch there are about 70K repositories. Some of them are reconciled to Wikidata (often directly reusing the Wikidata URI). For those that have a Wikidata URI we can retrieve coordinates by either using a federated approach or simply sending two consecutive queries (1st to artresearch to retrieve the Wikidata URIs and a 2nd to Wikidata, giving as input the list of URIs previously retrieved).


**1. Data extraction (SPARQL — artresearch)**  
We query the artresearch endpoint to retrieve all repositories that currently hold at least one artwork (`crm:P50_has_current_keeper`), along with the
count of works held by each. We filter to works of type `aat:300133025` (works of art) to exclude photographs and other record types.

Repositories in PHAROS are identified by local URIs and Wikidata URIs. We keep only those whose URI belongs to **Wikidata** (`wikidata.org/entity/Q...`), because Wikidata is our bridge to geographic coordinates. Repositories identified by local or other URIs are excluded at this stage.

**2. Coordinate lookup (SPARQL — Wikidata)**  
This is an example of **federated querying**: we use a second, completely independent SPARQL endpoint — the Wikidata Query Service — to enrich our data with geographic information that artresearch does not contain.

For each Wikidata QID, we retrieve the institution's coordinates
(`wdt:P625`) and English label. Since Wikidata rejects queries with too  many entities at once, we split QIDs into **batches of 50** and query them sequentially, with a short pause between requests to respect the endpoint's rate limits.

**3. Map visualization (plotly scatter_map)**  
Each dot on the map is a repository. Two visual variables encode the same quantity — number of works held — using redundant encoding for clarity:
- **Size**: larger dots hold more works
- **Color**: a Viridis gradient from dark (few works) to bright (many works)

Redundant encoding (using both size and color for the same variable) makes the map easier to read at a glance, especially where dots overlap.

In [1]:
import requests, time, pandas as pd
import plotly.express as px

sparql_endpoint = "https://artresearch.net/sparql"
headers = {"Accept": "application/sparql-results+json", "Cache-Control": "no-cache"}

query = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX custom: <https://artresearch.net/custom/>

SELECT DISTINCT ?repo (COUNT(?work) AS ?count)
WHERE {
  ?work a crm:E22_Human-Made_Object ;
        crm:P2_has_type <http://vocab.getty.edu/aat/300133025> ;
        crm:P50_has_current_keeper ?repo .
}
GROUP BY ?repo
ORDER BY DESC(?count)
"""

r = requests.get(sparql_endpoint, params={"query": query, "_": int(time.time())}, headers=headers)
bindings = r.json()['results']['bindings']
print(f"{len(bindings)} repositories found")

rows = [{
    "repo": b["repo"]["value"],
    "count": int(b["count"]["value"]),
    "qid": b["repo"]["value"].split("/")[-1] if "wikidata.org" in b["repo"]["value"] else None
} for b in bindings]

df = pd.DataFrame(rows)
df_wd = df.dropna(subset=["qid"]).copy()
print(f"{len(df_wd)} repositories with Wikidata URIs")

# --- get coordinates from Wikidata
def get_wikidata_coords(qids, batch_size=50):
    coords = {}
    batches = [qids[i:i+batch_size] for i in range(0, len(qids), batch_size)]
    print(f"Fetching {len(qids)} QIDs in {len(batches)} batches...")

    for i, batch in enumerate(batches):
        values = " ".join([f"wd:{qid}" for qid in batch])
        query = f"""
        SELECT ?item ?label ?lat ?lon WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:P625 ?coords .
          BIND(geof:latitude(?coords) AS ?lat)
          BIND(geof:longitude(?coords) AS ?lon)
          OPTIONAL {{ ?item rdfs:label ?label . FILTER(LANG(?label) = "en") }}
        }}
        """
        r = requests.get(
            "https://query.wikidata.org/sparql",
            params={"query": query},
            headers={
                "Accept": "application/sparql-results+json",
                "User-Agent": "artresearch-notebook/1.0 (mailto:you@example.com)"
            }
        )
        print(f"Batch {i+1}/{len(batches)}: status={r.status_code}")
        if r.status_code != 200:
            print(r.text[:200])
            continue

        for b in r.json()['results']['bindings']:
            qid = b['item']['value'].split("/")[-1]
            coords[qid] = {
                "lat": float(b['lat']['value']),
                "lon": float(b['lon']['value']),
                "label": b['label']['value'] if 'label' in b else qid
            }
        time.sleep(0.5)  # be polite to Wikidata

    return coords

qids = df_wd["qid"].unique().tolist()
print(f"Fetching coordinates for {len(qids)} repositories...")
coords = get_wikidata_coords(qids)
print(f"Got coordinates for {len(coords)} repositories")

df_wd["lat"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("lat"))
df_wd["lon"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("lon"))
df_wd["label"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("label", "unknown"))
df_wd = df_wd.dropna(subset=["lat", "lon"])
print(f"{len(df_wd)} repositories with coordinates")

fig = px.scatter_map(
    df_wd,
    lat="lat", lon="lon",
    size="count",
    color="count",
    hover_name="label",
    hover_data={"count": True, "lat": False, "lon": False},
    color_continuous_scale="Viridis",
    size_max=40,
    zoom=2,
    title="Repositories holding artworks (size = number of works)"
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

75118 repositories found
1292 repositories with Wikidata URIs
Fetching coordinates for 1292 repositories...
Fetching 1292 QIDs in 26 batches...
Batch 1/26: status=200
Batch 2/26: status=200
Batch 3/26: status=200
Batch 4/26: status=200
Batch 5/26: status=200
Batch 6/26: status=200
Batch 7/26: status=200
Batch 8/26: status=200
Batch 9/26: status=200
Batch 10/26: status=200
Batch 11/26: status=200
Batch 12/26: status=200
Batch 13/26: status=200
Batch 14/26: status=200
Batch 15/26: status=200
Batch 16/26: status=200
Batch 17/26: status=200
Batch 18/26: status=200
Batch 19/26: status=200
Batch 20/26: status=200
Batch 21/26: status=200
Batch 22/26: status=200
Batch 23/26: status=200
Batch 24/26: status=200
Batch 25/26: status=200
Batch 26/26: status=200
Got coordinates for 670 repositories
670 repositories with coordinates


- **Geographic concentration**: most artworks held by a small number of large institutions. A map dominated by a few very large dots suggests high concentration.
- **Regional bias**: Gaps on the map reveal regions whose institutions are either not collecting, not digitizing, or not connected to international photo archive networks like PHAROS.
- **Institutional hierarchies**: if you hover over the largest dots you'll likely see the institutions you would expect.
- **Limitations of the data**: repositories without a Wikidata URI
  and therefore without coordinates — are invisible on this map. The absence of a dot does not mean no works are held there; it may simply mean the institution is not yet linked to Wikidata.

This visualization reveal structural patterns in how art history has been documented, collected, and made accessible across the world.

For instance, it is clear that art collections in scope in artresearch are mostly preserved in Europe and North America. While this doesn't tell us about the type of artworks and their provenance, it clearly indicates the representativeness of the artresearch dataset wrt to European and, partially, Eastern American Heritage.

## 💡 Plot distribution of artworks by repository and by archive

The PHAROS consortium brings together major photo archives from across Europe and North America. Each archive has its own history,
institutional mandate, and geographic focus — the Zeri Foundation
specializes in Italian art, the PMC in British art, the RKD in Dutch and Flemish art. But how visible are these differences in the data?

We build on top of the previous chart to identify not only the repositories location, but to appreciate which institution focuses on which areas the most. This map asks: **do different photo archives document artworks held by geographically distinct sets of institutions?** By plotting repositories on a map and coloring them by the archive that documented them, we can see at a glance whether each archive has a distinct geographic footprint
— or whether they all converge on the same handful of major museums.

The previous map showed repositories colored by **number of works** — a quantitative variable encoded with a continuous color scale. This map replaces that with **archive name** — a categorical variable encoded with a discrete color palette, one color per archive.

This is a deliberate design choice: when comparing groups (archives), categorical color is more effective than a gradient because it makes group membership immediately visible without requiring the viewer to read a color scale. Node size still encodes the number of works, preserving the quantitative dimension.

The SPARQL query adds one variable — `?archive` — and one filter
(`STRSTARTS` on the e31 namespace) to restrict results to the eight known PHAROS institutional datasets. Everything else in the pipeline is identical to the previous map.


**1. Data extraction (SPARQL — artresearch)**  
We query for all `(repository, archive)` pairs, counting how many works each archive documents at each repository. A repository can appear multiple times — once per archive that documents works held there — producing overlapping dots on the map for heavily documented institutions.

**2. Coordinate lookup (SPARQL — Wikidata)**  
Identical to the previous map: federated querying in batches of 50 QIDs, retrieving coordinates and English labels from the Wikidata Query Service.

**3. Map visualization (plotly scatter_map)**  
- **Color** encodes archive — one distinct color per institution
- **Size** encodes number of works documented by that archive at that repository
- **Hover** shows the repository name, archive, and work count

In [ ]:
import requests, time, pandas as pd
import plotly.express as px

sparql_endpoint = "https://artresearch.net/sparql"
headers = {"Accept": "application/sparql-results+json", "Cache-Control": "no-cache"}

query = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX custom: <https://artresearch.net/custom/>

SELECT DISTINCT ?repo ?archive (COUNT(?work) AS ?count)
WHERE {
  ?work a crm:E22_Human-Made_Object ;
        crm:P2_has_type <http://vocab.getty.edu/aat/300133025> ;
        crm:P50_has_current_keeper ?repo ;
        crm:P70i_is_documented_in ?archive .
  FILTER(STRSTARTS(STR(?archive), "https://artresearch.net/resource/e31/"))
}
GROUP BY ?repo ?archive
ORDER BY DESC(?count)
"""

r = requests.get(sparql_endpoint, params={"query": query, "_": int(time.time())}, headers=headers)
bindings = r.json()['results']['bindings']
print(f"{len(bindings)} results found")

rows = [{
    "repo": b["repo"]["value"],
    "archive": b["archive"]["value"].split("/")[-1],
    "count": int(b["count"]["value"]),
    "qid": b["repo"]["value"].split("/")[-1] if "wikidata.org" in b["repo"]["value"] else None
} for b in bindings]

df = pd.DataFrame(rows)
df_wd = df.dropna(subset=["qid"]).copy()
print(f"{len(df_wd)} repositories with Wikidata URIs")

def get_wikidata_coords(qids, batch_size=50):
    coords = {}
    batches = [qids[i:i+batch_size] for i in range(0, len(qids), batch_size)]
    print(f"Fetching {len(qids)} QIDs in {len(batches)} batches...")

    for i, batch in enumerate(batches):
        values = " ".join([f"wd:{qid}" for qid in batch])
        query = f"""
        SELECT ?item ?label ?lat ?lon WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:P625 ?coords .
          BIND(geof:latitude(?coords) AS ?lat)
          BIND(geof:longitude(?coords) AS ?lon)
          OPTIONAL {{ ?item rdfs:label ?label . FILTER(LANG(?label) = "en") }}
        }}
        """
        r = requests.get(
            "https://query.wikidata.org/sparql",
            params={"query": query},
            headers={
                "Accept": "application/sparql-results+json",
                "User-Agent": "artresearch-notebook/1.0 (mailto:you@example.com)"
            }
        )
        print(f"Batch {i+1}/{len(batches)}: status={r.status_code}")
        if r.status_code != 200:
            print(r.text[:200])
            continue
        for b in r.json()['results']['bindings']:
            qid = b['item']['value'].split("/")[-1]
            coords[qid] = {
                "lat": float(b['lat']['value']),
                "lon": float(b['lon']['value']),
                "label": b['label']['value'] if 'label' in b else qid
            }
        time.sleep(0.5)

    return coords

qids = df_wd["qid"].unique().tolist()
coords = get_wikidata_coords(qids)
print(f"Got coordinates for {len(coords)} repositories")

df_wd["lat"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("lat"))
df_wd["lon"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("lon"))
df_wd["label"] = df_wd["qid"].map(lambda q: coords.get(q, {}).get("label", "unknown"))
df_wd = df_wd.dropna(subset=["lat", "lon"])
print(f"{len(df_wd)} repositories with coordinates")

fig = px.scatter_map(
    df_wd,
    lat="lat", lon="lon",
    size="count",
    color="archive",                          # one color per archive
    hover_name="label",
    hover_data={"count": True, "archive": True, "lat": False, "lon": False},
    size_max=40,
    zoom=2,
    title="Repositories by Photo Archive (color = archive, size = number of works)"
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

78993 results found
2246 repositories with Wikidata URIs
Fetching 1292 QIDs in 26 batches...
Batch 1/26: status=200
Batch 2/26: status=200
Batch 3/26: status=200


This visualization extends the previous map by adding a **comparative dimension**: instead of asking "where are the works?", we now ask "who is looking at them?" — a question about the sociology of art historical documentation as much as about geography.

- **Geographic specialization**: certain archives cluster in specific regions, but they mostly overlap across the whole map. A map where colors are mixed everywhere suggests broad, overlapping coverage. It seems clear that the scope of institutions overlap significantly. Notably, Marburg photo archive is the only one that includes a small but significant collection from Middle East, which seems to be overlooked in other collections.
- **Shared repositories**: when multiple colored dots overlap at the same location, it means several archives all document works at that institution — typically a major museum like the Louvre or the Met, which every archive  has reason to cover.
- **Peripheral coverage**: small dots far from the major clusters reveal institutions that are documented by only one archive and hold relatively few works — niche collections that would be invisible without that archive's specific focus. For instance, Zeri is the institute that includes art from the most diverse museums in the US, although significantly smaller, which are not exclusively from the East coast like other archives document.
- **Asymmetries in collection size**: even within the same geographic area, dot sizes can vary dramatically between archives, revealing that some archives document a few institutions very deeply while others spread coverage more evenly. For instance, the three German institutions (Hertziana, Marburg, KHI) belong to the same consortium. Notice that Hertziana and KHI collections overlap significantly in terms of locations (mostly Italy, France and Germany) and dimentions. Marburg collection seems to complement the former two with more artworks from the German speaking countries, while focusing less on the Italian heritage.

PMC and Warburg institute are currently missing since their repositories have not been reconciled yet.

**Limitations of the analysis**
- Notice that not all repositories have Wikidata reconciliation and therefore were not plotted.
- Notably, only the most representative museums and institutions (i.e. the ones with the highest number of artworks) are reconciled.
- We can conclude that the analysis is limited in scope, although we can claim representativeness, since the reconciliation policy gave priority to the most populated museums.
- Let alone that the potentially long tail of smaller and peripheral museums is not represented.
- Moreover, it's worth noting that a high number of artworks described in artresearch belongs to private collections, whose location cannot be shared for privacy reasons, or their location is unknown.



# Topical analysis: co-occurrence of artists in museum collections

Do certain artists always appear together in museums and collections?
If every institution that holds a Rembrandt also holds a Vermeer, that is not
a coincidence — it reflects shared historical, stylistic, or market dynamics
that shaped how collections were built.

## 💡 Artist Co-occurrence in Repositories

Co-occurrence is a simple but powerful idea. In our example below, two artists **co-occur** in a
repository if they both have works held by the same institution. If Rembrandt
and Vermeer both have paintings in the Rijksmuseum, they co-occur there.
If they also both appear in the Louvre, their co-occurrence weight increases.

The more repositories two artists share, the **stronger their connection**.

This is a form of **implicit relationship extraction**: we never explicitly
said "Rembrandt and Vermeer are related", but by observing that institutions
repeatedly collect them together, we can infer a meaningful association —
whether stylistic, historical, or geographic.

**Association rule mining** originates from market basket analysis: given a
supermarket dataset of purchases, find products that are frequently bought
together. The classic example: *customers who buy bread and butter also tend
to buy milk*.

We apply the same logic to cultural heritage data:
- A **transaction** is a repository (museum, gallery, collection)
- An **item** is an artist whose works are held by that repository
- A **rule** `A → B` means: repositories that hold works by artist A
  also tend to hold works by artist B

Two metrics control which rules we keep:

- **Support**: the fraction of repositories where both artists appear.
  Low support means the pair is rare; high support means they co-occur
  frequently across many institutions.
- **Confidence**: the fraction of repositories with A that *also* have B.
  Confidence = 1.0 means a perfect implication — every repo with A has B.
  We use **confidence = 0.9**, meaning at least 90% of repos holding A
  also hold B. This allows for a small number of exceptions while still
  capturing strong systematic associations.


**1. Data extraction (SPARQL)**  
We first identify the **top 50 most collected artists** in artresearch by number
of works. We restrict the number of artists to those identified by ULAN URI (which increases the chances the artist appears in several archives and repositories). This is needed to produce a readable network at the end. We then fetch all `(artist, repository)` pairs for those artists, restricted to repositories identified by a Wikidata URI.

**2. Transaction encoding (mlxtend)**  
We restructure the data so that each **row is a repository** and each  **column is an artist**, with `True/False` values indicating presence. This is the standard input format for the Apriori algorithm.

**3. Association rule mining (Apriori)**  
We run the Apriori algorithm with a low minimum support threshold (5% of repositories) to find all frequent artist combinations. We then filter the resulting rules to keep only those with **confidence ≥ 0.9** and a single antecedent (pairwise rules only), producing a focused set of strong
directional associations.

**4. Network visualization (networkx + plotly)**  
Rules become **directed edges** in a graph: an arrow from A to B means "90%+ of repos with A also have B". Node size and color encode degree — artists that appear in many rules are more central to the network.


In [ ]:
!pip install mlxtend --break-system-packages -q
import requests, time, pandas as pd
from itertools import combinations
import networkx as nx
import plotly.graph_objects as go
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [16]:
sparql_endpoint = "https://artresearch.net/sparql"
headers = {"Accept": "application/sparql-results+json", "Cache-Control": "no-cache"}

# --- Query 1: top 50 artists
query = """
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX pharos-meta: <https://artresearch.net/resource/pharos/vocab/meta/>

SELECT ?artist (SAMPLE(?name) AS ?label) (COUNT(DISTINCT ?work) AS ?workcount)
WHERE {
  ?work a crm:E22_Human-Made_Object ;
        crm:P2_has_type <http://vocab.getty.edu/aat/300133025> ;
        crm:P108i_was_produced_by ?production ;
        crm:P50_has_current_keeper ?repo .
  ?production crm:P14_carried_out_by ?artist .
  FILTER(STRSTARTS(STR(?artist), "http://vocab.getty.edu/ulan/"))
  FILTER(STRSTARTS(STR(?repo), "http://www.wikidata.org/entity/"))
  OPTIONAL {
    ?artist crm:P1_is_identified_by ?app .
    ?app crm:P2_has_type/crm:P127_has_broader_term* pharos-meta:preferred_name ;
         crm:P190_has_symbolic_content ?name .
  }
}
GROUP BY ?artist
ORDER BY DESC(?workcount)
LIMIT 50
"""

r = requests.get(sparql_endpoint, params={"query": query, "_": int(time.time())}, headers=headers)
bindings_q1 = r.json()['results']['bindings']
top_artists = set(b["artist"]["value"] for b in bindings_q1)
artist_labels = {b["artist"]["value"]: b["label"]["value"] if "label" in b else b["artist"]["value"].split("/")[-1] for b in bindings_q1}
print(f"{len(top_artists)} top artists")

# --- Query 2: artist-repo pairs for top artists
values_clause = "VALUES ?artist { " + " ".join([f"<{uri}>" for uri in top_artists]) + " }"

query2 = f"""
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>

SELECT ?artist ?repo
WHERE {{
  {values_clause}
  ?production crm:P14_carried_out_by ?artist .
  ?work crm:P108i_was_produced_by ?production ;
        a crm:E22_Human-Made_Object ;
        crm:P2_has_type <http://vocab.getty.edu/aat/300133025> ;
        crm:P50_has_current_keeper ?repo .
  FILTER(STRSTARTS(STR(?repo), "http://www.wikidata.org/entity/"))
}}
GROUP BY ?artist ?repo
"""

r = requests.get(sparql_endpoint, params={"query": query2, "_": int(time.time())}, headers=headers)
bindings = r.json()['results']['bindings']
print(f"{len(bindings)} artist-repo pairs")

# --- Build artist sets per repo (transactions = repositories, items = artists)
repo_artists = {}
for b in bindings:
    repo = b["repo"]["value"]
    artist = b["artist"]["value"]
    repo_artists.setdefault(repo, set()).add(artist)

transactions = [list(artists) for artists in repo_artists.values()]
print(f"{len(transactions)} repositories (transactions)")

# --- Apriori
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_enc = pd.DataFrame(te_array, columns=te.columns_)

# min_support = fraction of repos an artist must appear in
# set low (e.g. 0.05 = at least 5% of repos) to catch rare but perfectly correlated pairs
frequent_itemsets = apriori(df_enc, min_support=0.05, use_colnames=True)
print(f"{len(frequent_itemsets)} frequent itemsets")

# confidence=1 means: every repo with A also has B
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.8)
rules = rules[rules["antecedents"].apply(len) == 1]  # keep only pairs (A -> B)
print(f"{len(rules)} rules with confidence=1")
print(rules[["antecedents", "consequents", "support", "lift"]].head(10))

# --- Build network from rules
G = nx.DiGraph()  # directed: A -> B means every repo with A also has B
for _, row in rules.iterrows():
    a = list(row["antecedents"])[0]
    b = list(row["consequents"])[0]
    G.add_node(a, label=artist_labels.get(a, a.split("/")[-1]))
    G.add_node(b, label=artist_labels.get(b, b.split("/")[-1]))
    G.add_edge(a, b, support=row["support"], lift=row["lift"])

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

pos = nx.spring_layout(G, seed=42, k=2)



import plotly.graph_objects as go

edge_traces = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    # draw arrow as annotation, line as trace
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode="lines",
        line=dict(width=1, color="#aaa"),
        hoverinfo="none"
    ))

# arrows via layout annotations
annotations = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    annotations.append(dict(
        ax=x0, ay=y0,
        x=x1, y=y1,
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=2,
        arrowsize=1.5,
        arrowwidth=1,
        arrowcolor="#888"
    ))

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_labels = [G.nodes[n]["label"] for n in G.nodes()]
node_degree = [G.degree(n) for n in G.nodes()]

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode="markers+text",
    text=node_labels,
    textposition="top center",
    hoverinfo="text",
    marker=dict(
        size=[10 + d * 4 for d in node_degree],
        color=node_degree,
        colorscale="Viridis",
        colorbar=dict(title="Degree"),
        line=dict(width=1, color="white")
    )
)

fig = go.Figure(
    data=edge_traces + [node_trace],
    layout=go.Layout(
        title="Artist Co-occurrence Network (Apriori, confidence=0.9)",
        showlegend=False,
        hovermode="closest",
        annotations=annotations,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=800
    )
)
fig.show()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

50 top artists


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

1875 artist-repo pairs
246 repositories (transactions)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

6665 frequent itemsets


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

61 rules with confidence=1
                               antecedents  \
0  (http://vocab.getty.edu/ulan/500005036)   
1  (http://vocab.getty.edu/ulan/500016881)   
2  (http://vocab.getty.edu/ulan/500002921)   
3  (http://vocab.getty.edu/ulan/500115190)   
4  (http://vocab.getty.edu/ulan/500115200)   
5  (http://vocab.getty.edu/ulan/500115366)   
6  (http://vocab.getty.edu/ulan/500115390)   
7  (http://vocab.getty.edu/ulan/500115509)   
8  (http://vocab.getty.edu/ulan/500118936)   
9  (http://vocab.getty.edu/ulan/500005036)   

                               consequents   support      lift  
0  (http://vocab.getty.edu/ulan/500002921)  0.093496  5.051786  
1  (http://vocab.getty.edu/ulan/500002921)  0.073171  5.535000  
2  (http://vocab.getty.edu/ulan/500031075)  0.130081  3.027692  
3  (http://vocab.getty.edu/ulan/500002921)  0.109756  5.189062  
4  (http://vocab.getty.edu/ulan/500002921)  0.089431  5.011111  
5  (http://vocab.getty.edu/ulan/500002921)  0.065041  5.788235  
6  (http://

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future v

- **Directed edges reveal asymmetry**: if A → B but not B → A, it means
  B is more widely collected than A — B appears in many repos regardless,
  while A is only collected where B already is
- **Clusters** reveal groups of artists that institutions treat as a
  coherent unit — perhaps a national school, a period, or a market segment
- **Isolated nodes** are artists whose collecting pattern is independent
  from others in the top 50 — unique or cross-cutting figures
- **High-degree nodes** are artists that act as "anchors" of institutional
  collecting — their presence in a collection reliably predicts many others

Compared to simple co-occurrence counting, association rules give us
**directional, probabilistic relationships** — a much richer picture of
how collecting practices structure the art historical canon.